# 04-shiraz-finetune

Account D. Shiraz, a small Persian MobileBERT. Its own tokenizer. This one is the most likely to finish an epoch on a free Colab GPU.

Target classes: `recommended`, `not_recommended`, `no_idea`. The only model input is `body`. The only label is `recommendation_status`. The notebook reads those two columns from the preprocessed CSV and no other field.

This file is one of four. Run each file on its own Colab account at the same time:

- `01-parsbert-finetune.ipynb`
- `02-xlmr-finetune.ipynb`
- `03-tookabert-finetune.ipynb`
- `04-shiraz-finetune.ipynb`

Upload one zip from this machine: `data/processed/digikala-comments-preprocessed.csv.zip`. The archive holds `digikala-comments-preprocessed.csv`. Do not upload the raw comments file, the cleaned CSV, or the tokenized parquet. Those token ids belong to ParsBERT and the other three models cannot read them.

The notebook downloads that zip from the shared Drive file, then unpacks it on the Colab disk:

`https://drive.google.com/file/d/1vCkaGbSrXtlKUsdOrguUtuEYobT5vFxr/view`

Each notebook writes its own folder under `MyDrive/digikala-experiments/runs/` and `MyDrive/digikala-experiments/best/`. Stage 1 trains only the classifier. Stage 2 unfreezes the last encoder block. Validation macro F1 picks the checkpoint. Test accuracy, precision, recall, and F1 are written once, into that model's `best` folder.

Do not use `HooshvareLab/bert-fa-base-uncased-sentiment-digikala`.


In [1]:
MODEL_NAME = "lifeweb-ai/shiraz"
RUN_STAGE = "both"  # "head", "last_block", or "both"
MAX_LENGTH = 128
EPOCHS = 1
BATCH_SIZE = 16
GRAD_ACCUM = 2
HEAD_LR = 1e-3
BLOCK_LR = 2e-5
HEAD_LR_WHEN_UNFROZEN = 1e-4
SEED = 2026

DRIVE_DIR = "/content/drive/MyDrive/digikala-experiments"
DRIVE_FILE_ID = "1vCkaGbSrXtlKUsdOrguUtuEYobT5vFxr"
DATA_ZIP = "/content/data/digikala-comments-preprocessed.csv.zip"
LOCAL_CSV = "/content/data/digikala-comments-preprocessed.csv"

LABELS = ["recommended", "not_recommended", "no_idea"]
LABEL_CODE = {label: code for code, label in enumerate(LABELS)}
SPLIT_SEED = 2026
TRAIN_SHARE = 0.70
VALIDATION_SHARE = 0.15
TEST_SHARE = 0.15
SPLITS = ["train", "validation", "test"]
SPLIT_SHARE = {"train": TRAIN_SHARE, "validation": VALIDATION_SHARE, "test": TEST_SHARE}


## Drive and the zipped CSV

Mount Drive for the experiment logs. Download the shared zip by file id, unpack it on the Colab disk, and read only `body` and `recommendation_status`.


In [2]:
from pathlib import Path
import json
import zipfile

import numpy as np
import polars as pl
import torch
from google.colab import drive
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

import gdown

drive.mount("/content/drive")
zip_path = Path(DATA_ZIP)
zip_path.parent.mkdir(parents=True, exist_ok=True)
if not zip_path.is_file():
    gdown.download(id=DRIVE_FILE_ID, output=str(zip_path), quiet=False)
if not zip_path.is_file():
    raise FileNotFoundError(
        f"Could not download Drive file {DRIVE_FILE_ID} to {zip_path}."
    )
with zipfile.ZipFile(zip_path) as archive:
    member = "digikala-comments-preprocessed.csv"
    names = [
        name
        for name in archive.namelist()
        if name.endswith(".csv") and not name.startswith("__MACOSX")
    ]
    if member not in names:
        if len(names) != 1:
            raise FileNotFoundError(f"Expected one CSV in the zip, found {names}")
        member = names[0]
    info = archive.getinfo(member)
    local_csv = Path(LOCAL_CSV)
    local_csv.parent.mkdir(parents=True, exist_ok=True)
    if (not local_csv.is_file()) or local_csv.stat().st_size != info.file_size:
        print("extracting", member, info.file_size, "bytes")
        with archive.open(info) as source, local_csv.open("wb") as target:
            while True:
                chunk = source.read(8 * 1024 * 1024)
                if not chunk:
                    break
                target.write(chunk)
drive_root = Path(DRIVE_DIR)
(drive_root / "runs").mkdir(parents=True, exist_ok=True)
(drive_root / "logs").mkdir(parents=True, exist_ok=True)
(drive_root / "best").mkdir(parents=True, exist_ok=True)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("zip:", zip_path, "bytes", zip_path.stat().st_size)
print("csv:", local_csv, "bytes", local_csv.stat().st_size)


Mounted at /content/drive


Downloading...
From (original): https://drive.google.com/uc?id=1vCkaGbSrXtlKUsdOrguUtuEYobT5vFxr
From (redirected): https://drive.google.com/uc?id=1vCkaGbSrXtlKUsdOrguUtuEYobT5vFxr&confirm=t&uuid=d861c8af-7fb5-4056-ad5c-b16cd5c4cbf3
To: /content/data/digikala-comments-preprocessed.csv.zip
100%|██████████| 157M/157M [00:02<00:00, 69.7MB/s]


extracting digikala-comments-preprocessed.csv 587997582 bytes
gpu: cpu
zip: /content/data/digikala-comments-preprocessed.csv.zip bytes 157243783
csv: /content/data/digikala-comments-preprocessed.csv bytes 587997582


## Stratified split

Each class is shuffled with seed 2026 and cut 70/15/15 into train, validation, and test. The cut uses `recommendation_status` only.


In [3]:
def fold_digits_for_tokenizer(body: pl.Expr, model_name: str) -> pl.Expr:
    """XLM-R was trained on ASCII digits. ParsBERT and TookaBERT keep Persian digits."""
    if "xlm-roberta" not in model_name.lower():
        return body
    folded = body
    for persian_digit, ascii_digit in zip("۰۱۲۳۴۵۶۷۸۹", "0123456789"):
        folded = folded.str.replace_all(persian_digit, ascii_digit, literal=True)
    return folded


reviews = pl.read_csv(
    LOCAL_CSV,
    columns=["body", "recommendation_status"],
    schema_overrides={
        "body": pl.String,
        "recommendation_status": pl.String,
    },
)
reviews = reviews.with_columns(
    fold_digits_for_tokenizer(pl.col("body"), MODEL_NAME).alias("body")
)
labeled = reviews.filter(
    pl.col("body").is_not_null()
    & (pl.col("body").str.strip_chars() != "")
    & pl.col("recommendation_status").is_in(LABELS)
).with_columns(
    pl.col("recommendation_status").replace_strict(LABEL_CODE).alias("label_id")
)

parts = []
for label in LABELS:
    group = labeled.filter(pl.col("recommendation_status") == label)
    shuffled = group.sample(fraction=1.0, shuffle=True, seed=SPLIT_SEED)
    count = shuffled.height
    n_train = int(count * TRAIN_SHARE)
    n_validation = int(count * VALIDATION_SHARE)
    n_test = count - n_train - n_validation
    parts.append(
        shuffled.with_columns(
            pl.Series(
                "split",
                (["train"] * n_train)
                + (["validation"] * n_validation)
                + (["test"] * n_test),
            )
        )
    )
labeled = pl.concat(parts)
print(
    labeled.group_by("split", "recommendation_status")
    .len()
    .sort("split", "recommendation_status")
)


shape: (9, 3)
┌────────────┬───────────────────────┬─────────┐
│ split      ┆ recommendation_status ┆ len     │
│ ---        ┆ ---                   ┆ ---     │
│ str        ┆ str                   ┆ u32     │
╞════════════╪═══════════════════════╪═════════╡
│ test       ┆ no_idea               ┆ 89048   │
│ test       ┆ not_recommended       ┆ 75671   │
│ test       ┆ recommended           ┆ 624035  │
│ train      ┆ no_idea               ┆ 415555  │
│ train      ┆ not_recommended       ┆ 353125  │
│ train      ┆ recommended           ┆ 2912158 │
│ validation ┆ no_idea               ┆ 89047   │
│ validation ┆ not_recommended       ┆ 75669   │
│ validation ┆ recommended           ┆ 624034  │
└────────────┴───────────────────────┴─────────┘


## Classifier head, then the last encoder block

Stage `head` trains only the new classification layer. Stage `last_block` starts from those weights, unfreezes the top encoder block, and uses a smaller learning rate on that block than on the classifier.


In [4]:
class ReviewSet(Dataset):
    def __init__(self, frame: pl.DataFrame, tokenizer, max_length: int):
        self.texts = frame.get_column("body").to_list()
        self.labels = frame.get_column("label_id").to_list()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        encoded = self.tokenizer(
            self.texts[index] or "",
            truncation=True,
            max_length=self.max_length,
        )
        encoded["labels"] = int(self.labels[index])
        return encoded


def metric_bundle(actual, predicted):
    precision, recall, f1, _ = precision_recall_fscore_support(
        actual, predicted, labels=[0, 1, 2], average="macro", zero_division=0
    )
    per_precision, per_recall, per_f1, per_support = precision_recall_fscore_support(
        actual, predicted, labels=[0, 1, 2], average=None, zero_division=0
    )
    report = {
        "accuracy": float(accuracy_score(actual, predicted)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }
    for index, name in enumerate(LABELS):
        report[f"{name}_precision"] = float(per_precision[index])
        report[f"{name}_recall"] = float(per_recall[index])
        report[f"{name}_f1"] = float(per_f1[index])
        report[f"{name}_support"] = int(per_support[index])
    return report


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predicted = np.argmax(logits, axis=-1)
    scores = metric_bundle(labels, predicted)
    return {key: scores[key] for key in ("accuracy", "precision", "recall", "f1")}


def encoder_prefix(model) -> str:
    for prefix in ("bert", "roberta", "distilbert", "mobilebert"):
        if hasattr(model, prefix):
            return prefix
    raise AttributeError("This checkpoint has no bert/roberta/distilbert/mobilebert encoder.")


def freeze_encoder(model) -> None:
    prefix = encoder_prefix(model)
    for name, param in model.named_parameters():
        param.requires_grad = not name.startswith(prefix + ".")


def unfreeze_last_block(model) -> None:
    freeze_encoder(model)
    prefix = encoder_prefix(model)
    encoder = getattr(model, prefix).encoder
    layers = getattr(encoder, "layer", None)
    if layers is None:
        layers = encoder.layers
    for param in layers[-1].parameters():
        param.requires_grad = True
    for name, param in model.named_parameters():
        if "classifier" in name or name.startswith("score"):
            param.requires_grad = True


def parameter_groups(model, block_lr: float, head_lr: float):
    head, block = [], []
    prefix = encoder_prefix(model)
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith(prefix + "."):
            block.append(param)
        else:
            head.append(param)
    return [
        {"params": block, "lr": block_lr},
        {"params": head, "lr": head_lr},
    ]


class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        weight = None
        if self.class_weights is not None:
            weight = self.class_weights.to(outputs.logits.device)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=weight)
        return (loss, outputs) if return_outputs else loss


def slug(model_name: str) -> str:
    return model_name.replace("/", "--")


def write_run(stage: str, metrics: dict, model, tokenizer) -> Path:
    run_dir = Path(DRIVE_DIR) / "runs" / f"{slug(MODEL_NAME)}--{stage}"
    run_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(run_dir)
    tokenizer.save_pretrained(run_dir)
    payload = {"model_name": MODEL_NAME, "stage": stage, "max_length": MAX_LENGTH, **metrics}
    (run_dir / "metrics.json").write_text(json.dumps(payload, indent=2))
    log_path = Path(DRIVE_DIR) / "logs" / f"{slug(MODEL_NAME)}.jsonl"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a") as handle:
        handle.write(json.dumps(payload) + "\n")
    print("saved", run_dir)
    print(json.dumps({key: payload[key] for key in ("accuracy", "precision", "recall", "f1")}, indent=2))
    return run_dir


def promote_if_best(run_dir: Path, metrics: dict) -> None:
    best_dir = Path(DRIVE_DIR) / "best" / slug(MODEL_NAME)
    best_dir.mkdir(parents=True, exist_ok=True)
    best_metrics_path = best_dir / "metrics.json"
    previous = -1.0
    if best_metrics_path.is_file():
        previous = json.loads(best_metrics_path.read_text()).get("f1", -1.0)
    if metrics["f1"] <= previous:
        print("validation macro F1", round(metrics["f1"], 4), "does not beat", round(previous, 4))
        return
    for source in run_dir.iterdir():
        target = best_dir / source.name
        target.write_bytes(source.read_bytes())
    print("best model updated", best_dir, "macro F1", round(metrics["f1"], 4))


## Train

`RUN_STAGE = "both"` trains the head, reloads that checkpoint, then trains the last block. Change `MODEL_NAME` on the other Colab account. Do not run two stages of the same checkpoint on two accounts at once, or they will overwrite the same Drive folder.


In [5]:
torch.manual_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_frame = labeled.filter(pl.col("split") == "train")
valid_frame = labeled.filter(pl.col("split") == "validation")
test_frame = labeled.filter(pl.col("split") == "test")
print("train", train_frame.height, "validation", valid_frame.height, "test", test_frame.height)

class_weights = torch.tensor(
    compute_class_weight(
        "balanced",
        classes=np.array([0, 1, 2]),
        y=train_frame.get_column("label_id").to_numpy(),
    ),
    dtype=torch.float,
)
train_set = ReviewSet(train_frame, tokenizer, MAX_LENGTH)
valid_set = ReviewSet(valid_frame, tokenizer, MAX_LENGTH)
collator = DataCollatorWithPadding(tokenizer)


def make_args(stage: str, learning_rate: float) -> TrainingArguments:
    return TrainingArguments(
        output_dir=f"/content/checkpoints/{slug(MODEL_NAME)}-{stage}",
        learning_rate=learning_rate,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        logging_steps=200,
        report_to="none",
        seed=SEED,
        dataloader_num_workers=2,
    )


def score_frame(model, frame: pl.DataFrame) -> dict:
    holder = ReviewSet(frame, tokenizer, MAX_LENGTH)
    args = TrainingArguments(
        output_dir="/tmp/score",
        per_device_eval_batch_size=BATCH_SIZE,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )
    trainer = Trainer(model=model, args=args, data_collator=collator, tokenizer=tokenizer)
    prediction = trainer.predict(holder)
    return metric_bundle(prediction.label_ids, np.argmax(prediction.predictions, axis=-1))


def train_stage(stage: str, model) -> None:
    if stage == "head":
        freeze_encoder(model)
        learning_rate = HEAD_LR
        groups = None
    elif stage == "last_block":
        unfreeze_last_block(model)
        learning_rate = BLOCK_LR
        groups = parameter_groups(model, BLOCK_LR, HEAD_LR_WHEN_UNFROZEN)
    else:
        raise ValueError(stage)
    trainable = sum(param.numel() for param in model.parameters() if param.requires_grad)
    print(stage, "trainable parameters", trainable)
    trainer = WeightedTrainer(
        model=model,
        args=make_args(stage, learning_rate),
        train_dataset=train_set,
        eval_dataset=valid_set,
        data_collator=collator,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
        processing_class=tokenizer,
    )
    if groups is not None:
        def create_optimizer():
            trainer.optimizer = torch.optim.AdamW(groups, weight_decay=0.01)
            return trainer.optimizer
        trainer.create_optimizer = create_optimizer
    trainer.train()
    metrics = score_frame(model, valid_frame)
    metrics["split"] = "validation"
    run_dir = write_run(stage, metrics, model, tokenizer)
    promote_if_best(run_dir, metrics)


stages = ["head", "last_block"] if RUN_STAGE == "both" else [RUN_STAGE]
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={index: name for index, name in enumerate(LABELS)},
    label2id=LABEL_CODE,
)
for stage in stages:
    if stage == "last_block":
        head_dir = Path(DRIVE_DIR) / "runs" / f"{slug(MODEL_NAME)}--head"
        if not head_dir.is_dir():
            raise FileNotFoundError(f"Train the head stage first. Missing {head_dir}")
        model = AutoModelForSequenceClassification.from_pretrained(head_dir)
        tokenizer = AutoTokenizer.from_pretrained(head_dir)
        train_set = ReviewSet(train_frame, tokenizer, MAX_LENGTH)
        valid_set = ReviewSet(valid_frame, tokenizer, MAX_LENGTH)
        collator = DataCollatorWithPadding(tokenizer)
    train_stage(stage, model)

best_dir = Path(DRIVE_DIR) / "best" / slug(MODEL_NAME)
best_dir.mkdir(parents=True, exist_ok=True)
best_model = AutoModelForSequenceClassification.from_pretrained(best_dir)
tokenizer = AutoTokenizer.from_pretrained(best_dir)
test_metrics = score_frame(best_model, test_frame)
test_metrics["split"] = "test"
test_metrics["model_name"] = json.loads((best_dir / "metrics.json").read_text()).get("model_name")
(best_dir / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))
print("test", json.dumps({key: test_metrics[key] for key in ("accuracy", "precision", "recall", "f1")}, indent=2))


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/356 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

train 3680838 validation 788750 test 788754


model.safetensors: reconstructing file:   0%|          |  0.00B /  186MB            

... (5 outputs left out)

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.508060,0.598326,0.764541,0.573622,0.667409,0.604110


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

## Save the finished head

Run this after the head epoch finishes and scoring crashes. It writes the trained classifier to Drive and does not train again.


In [2]:
from pathlib import Path
import json
import shutil

stage = "head"
run_dir = Path(DRIVE_DIR) / "runs" / f"{slug(MODEL_NAME)}--{stage}"
best_dir = Path(DRIVE_DIR) / "best" / slug(MODEL_NAME)
run_dir.mkdir(parents=True, exist_ok=True)
best_dir.mkdir(parents=True, exist_ok=True)

local_dir = Path(f"/content/checkpoints/{slug(MODEL_NAME)}-{stage}")
checkpoints = []
if local_dir.is_dir():
    checkpoints = sorted(local_dir.glob("checkpoint-*"), key=lambda path: path.stat().st_mtime)
print("local checkpoints:", [path.name for path in checkpoints])

if "model" in globals() and "tokenizer" in globals():
    model.save_pretrained(run_dir)
    tokenizer.save_pretrained(run_dir)
    print("saved in-memory model to", run_dir)
elif checkpoints and (checkpoints[-1] / "config.json").is_file():
    source = checkpoints[-1]
    for item in source.iterdir():
        if item.is_file():
            shutil.copy2(item, run_dir / item.name)
    print("copied", source, "to", run_dir)
else:
    raise RuntimeError("No model in memory and no local checkpoint. The runtime was reset.")

for item in run_dir.iterdir():
    if item.is_file():
        shutil.copy2(item, best_dir / item.name)

if not (run_dir / "metrics.json").is_file():
    note = {
        "model_name": MODEL_NAME,
        "stage": stage,
        "source": "saved after the scoring crash",
    }
    (run_dir / "metrics.json").write_text(json.dumps(note, indent=2))
    (best_dir / "metrics.json").write_text(json.dumps(note, indent=2))
print("drive run:", run_dir)
print("drive best:", best_dir)
print("files:", sorted(path.name for path in run_dir.iterdir()))


NameError: name 'DRIVE_DIR' is not defined

## Score the saved head

Run this after the save cell. It loads the Drive copy and scores validation, then test. It does not train.


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

saved_dir = Path(DRIVE_DIR) / "best" / slug(MODEL_NAME)
saved_model = AutoModelForSequenceClassification.from_pretrained(saved_dir)
saved_tokenizer = AutoTokenizer.from_pretrained(saved_dir)
saved_collator = DataCollatorWithPadding(saved_tokenizer)


def score_saved(frame):
    holder = ReviewSet(frame, saved_tokenizer, MAX_LENGTH)
    args = TrainingArguments(
        output_dir="/tmp/score",
        per_device_eval_batch_size=BATCH_SIZE,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )
    trainer = Trainer(
        model=saved_model,
        args=args,
        data_collator=saved_collator,
        processing_class=saved_tokenizer,
    )
    prediction = trainer.predict(holder)
    return metric_bundle(prediction.label_ids, prediction.predictions.argmax(axis=-1))


validation_metrics = score_saved(valid_frame)
validation_metrics.update({"split": "validation", "stage": "head", "model_name": MODEL_NAME})
test_metrics = score_saved(test_frame)
test_metrics.update({"split": "test", "stage": "head", "model_name": MODEL_NAME})
(saved_dir / "metrics.json").write_text(json.dumps(validation_metrics, indent=2))
(saved_dir / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))
print("validation", {key: validation_metrics[key] for key in ("accuracy", "precision", "recall", "f1")})
print("test", {key: test_metrics[key] for key in ("accuracy", "precision", "recall", "f1")})


## Continue from the saved head

Run this after the save cell. It loads the head from Drive and trains the last encoder block. The block is written to Drive before validation and test are scored again.


In [ ]:
import shutil
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

head_dir = Path(DRIVE_DIR) / "runs" / f"{slug(MODEL_NAME)}--head"
local_dir = Path(f"/content/checkpoints/{slug(MODEL_NAME)}-head")
if not (head_dir / "config.json").is_file():
    checkpoints = sorted(local_dir.glob("checkpoint-*"), key=lambda path: path.stat().st_mtime) if local_dir.is_dir() else []
    if checkpoints and (checkpoints[-1] / "config.json").is_file():
        head_dir = checkpoints[-1]
    elif "model" in globals():
        head_dir = Path(DRIVE_DIR) / "runs" / f"{slug(MODEL_NAME)}--head"
        head_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(head_dir)
        tokenizer.save_pretrained(head_dir)
    else:
        raise FileNotFoundError(f"Missing the head checkpoint at {head_dir}")

torch.manual_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(head_dir)
tokenizer = AutoTokenizer.from_pretrained(head_dir)
train_set = ReviewSet(train_frame, tokenizer, MAX_LENGTH)
valid_set = ReviewSet(valid_frame, tokenizer, MAX_LENGTH)
collator = DataCollatorWithPadding(tokenizer)
print("continuing from", head_dir)

stage = "last_block"
unfreeze_last_block(model)
groups = parameter_groups(model, BLOCK_LR, HEAD_LR_WHEN_UNFROZEN)
trainable = sum(param.numel() for param in model.parameters() if param.requires_grad)
print(stage, "trainable parameters", trainable)
trainer = WeightedTrainer(
    model=model,
    args=make_args(stage, BLOCK_LR),
    train_dataset=train_set,
    eval_dataset=valid_set,
    data_collator=collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    processing_class=tokenizer,
)


def create_optimizer():
    trainer.optimizer = torch.optim.AdamW(groups, weight_decay=0.01)
    return trainer.optimizer


trainer.create_optimizer = create_optimizer
trainer.train()

block_dir = Path(DRIVE_DIR) / "runs" / f"{slug(MODEL_NAME)}--{stage}"
block_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(block_dir)
tokenizer.save_pretrained(block_dir)
print("saved last block before scoring", block_dir)


def score_block(frame):
    holder = ReviewSet(frame, tokenizer, MAX_LENGTH)
    args = TrainingArguments(
        output_dir="/tmp/score-last-block",
        per_device_eval_batch_size=BATCH_SIZE,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )
    scorer = Trainer(
        model=model,
        args=args,
        data_collator=collator,
        processing_class=tokenizer,
    )
    prediction = scorer.predict(holder)
    return metric_bundle(prediction.label_ids, prediction.predictions.argmax(axis=-1))


validation_metrics = score_block(valid_frame)
validation_metrics.update({"split": "validation", "stage": stage, "model_name": MODEL_NAME})
(block_dir / "metrics.json").write_text(json.dumps(validation_metrics, indent=2))
best_dir = Path(DRIVE_DIR) / "best" / slug(MODEL_NAME)
best_dir.mkdir(parents=True, exist_ok=True)
previous = -1.0
if (best_dir / "metrics.json").is_file():
    previous = json.loads((best_dir / "metrics.json").read_text()).get("f1", -1.0)
if validation_metrics["f1"] > previous:
    for item in block_dir.iterdir():
        if item.is_file():
            shutil.copy2(item, best_dir / item.name)
    print("best model updated", best_dir, "macro F1", round(validation_metrics["f1"], 4))
else:
    print("validation macro F1", round(validation_metrics["f1"], 4), "does not beat", round(previous, 4))

test_metrics = score_block(test_frame)
test_metrics.update({"split": "test", "stage": stage, "model_name": MODEL_NAME})
(block_dir / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))
if validation_metrics["f1"] > previous:
    (best_dir / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))
log_path = Path(DRIVE_DIR) / "logs" / f"{slug(MODEL_NAME)}.jsonl"
log_path.parent.mkdir(parents=True, exist_ok=True)
with log_path.open("a") as handle:
    handle.write(json.dumps(validation_metrics) + "\n")
    handle.write(json.dumps(test_metrics) + "\n")
print("validation", {key: validation_metrics[key] for key in ("accuracy", "precision", "recall", "f1")})
print("test", {key: test_metrics[key] for key in ("accuracy", "precision", "recall", "f1")})


## Push the checkpoint to MLflow

Run this while the Colab runtime is still alive, including when the disconnect warning appears. A hard shutdown cannot upload anything. This sends the newest Trainer checkpoint, including optimizer state and the last epoch, plus the saved Drive model files. The review CSV stays on Drive.


In [1]:
import os
import subprocess
from pathlib import Path
import json

subprocess.check_call(["pip", "install", "-q", "mlflow"])
import mlflow

MLFLOW_TRACKING_URI = "http://46.249.99.130:5000"
MLFLOW_EXPERIMENT = "04-shiraz-finetune"
MLFLOW_TRACKING_USERNAME = "amir"
MLFLOW_TRACKING_PASSWORD = "3vuy3wgbfiu3whbrifu3hwiorfujh3newikfjrni"
os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_TRACKING_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_TRACKING_PASSWORD

def checkpoint_step(path: Path) -> int:
    try:
        return int(path.name.split("-")[-1])
    except ValueError:
        return -1

found = []
checkpoint_root = Path("/content/checkpoints")
if checkpoint_root.is_dir():
    for stage_dir in checkpoint_root.iterdir():
        if not stage_dir.is_dir():
            continue
        for checkpoint in stage_dir.glob("checkpoint-*"):
            if (checkpoint / "trainer_state.json").is_file():
                found.append(checkpoint)
if not found:
    raise FileNotFoundError("No Trainer checkpoint under /content/checkpoints.")
checkpoint = max(found, key=checkpoint_step)
stage_name = checkpoint.parent.name
stage = "last_block" if stage_name.endswith("last_block") else "head"
state = json.loads((checkpoint / "trainer_state.json").read_text())
print("uploading", checkpoint, "step", state.get("global_step"), "epoch", state.get("epoch"))

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
with mlflow.start_run(run_name=f"{MLFLOW_EXPERIMENT}--{stage}"):
    mlflow.set_tags({
        "notebook": MLFLOW_EXPERIMENT,
        "model_name": MODEL_NAME,
        "model_slug": slug(MODEL_NAME),
        "stage": stage,
    })
    mlflow.log_params({
        "global_step": state.get("global_step"),
        "epoch": state.get("epoch"),
        "max_steps": state.get("max_steps"),
    })
    mlflow.log_artifacts(str(checkpoint), artifact_path="checkpoint")
    for folder_name in ("runs", "best"):
        drive_folder = Path(DRIVE_DIR) / folder_name
        if not drive_folder.is_dir():
            continue
        for child in drive_folder.iterdir():
            if child.is_dir() and slug(MODEL_NAME) in child.name:
                mlflow.log_artifacts(str(child), artifact_path=f"drive/{folder_name}/{child.name}")
                print("uploaded", child)
    run = mlflow.active_run()
    print("mlflow run", run.info.run_id)


FileNotFoundError: No Trainer checkpoint under /content/checkpoints.

## Pull the checkpoint from MLflow and continue

On a fresh runtime, run the notebook from the top through the cell that defines `WeightedTrainer` and through the split. Skip the Train cell, then run this cell. A finished head epoch continues into the last encoder block. An unfinished epoch resumes at its saved step.


In [ ]:
import os
import json
import shutil
import subprocess
from pathlib import Path

subprocess.check_call(["pip", "install", "-q", "mlflow"])
import mlflow
from mlflow.tracking import MlflowClient
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

MLFLOW_TRACKING_URI = "http://46.249.99.130:5000"
MLFLOW_EXPERIMENT = "04-shiraz-finetune"
MLFLOW_TRACKING_USERNAME = "amir"
MLFLOW_TRACKING_PASSWORD = "3vuy3wgbfiu3whbrifu3hwiorfujh3newikfjrni"
os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_TRACKING_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_TRACKING_PASSWORD
for frame_name in ("train_frame", "valid_frame", "test_frame"):
    if frame_name not in globals():
        raise RuntimeError("Run the notebook through the split cell before this cell.")
if "WeightedTrainer" not in globals():
    raise RuntimeError("Run the cell that defines WeightedTrainer before this cell.")
if "class_weights" not in globals():
    class_weights = torch.tensor(
        compute_class_weight(
            "balanced",
            classes=np.array([0, 1, 2]),
            y=train_frame.get_column("label_id").to_numpy(),
        ),
        dtype=torch.float,
    )


def make_args(stage: str, learning_rate: float) -> TrainingArguments:
    return TrainingArguments(
        output_dir=f"/content/checkpoints/{slug(MODEL_NAME)}-{stage}",
        learning_rate=learning_rate,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        logging_steps=200,
        report_to="none",
        seed=SEED,
        dataloader_num_workers=2,
    )
if MLFLOW_TRACKING_USERNAME:
    os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_TRACKING_USERNAME
if MLFLOW_TRACKING_PASSWORD:
    os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_TRACKING_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT)
if experiment is None:
    raise FileNotFoundError(f"MLflow experiment {MLFLOW_EXPERIMENT} does not exist.")
runs = client.search_runs(
    [experiment.experiment_id],
    filter_string=f"tags.model_slug = '{slug(MODEL_NAME)}'",
    order_by=["attributes.start_time DESC"],
    max_results=1,
)
if not runs:
    raise FileNotFoundError(f"No MLflow run for {MODEL_NAME}.")
run = runs[0]
print("restoring run", run.info.run_id, "stage", run.data.tags.get("stage"))
restore_root = Path("/content/mlflow-restore")
if restore_root.exists():
    shutil.rmtree(restore_root)
local_checkpoint = Path(mlflow.artifacts.download_artifacts(
    run_id=run.info.run_id,
    artifact_path="checkpoint",
    dst_path=str(restore_root),
))
state = json.loads((local_checkpoint / "trainer_state.json").read_text())
stage = run.data.tags.get("stage", "head")
epoch = float(state.get("epoch") or 0)
global_step = int(state.get("global_step") or 0)
max_steps = int(state.get("max_steps") or 0)
finished = epoch >= EPOCHS - 1e-6 or (max_steps > 0 and global_step >= max_steps)
print("restored step", global_step, "epoch", epoch, "finished", finished)

def build_trainer(stage_name, model, tokenizer):
    if stage_name == "head":
        freeze_encoder(model)
        learning_rate = HEAD_LR
        groups = None
    elif stage_name == "last_block":
        unfreeze_last_block(model)
        learning_rate = BLOCK_LR
        groups = parameter_groups(model, BLOCK_LR, HEAD_LR_WHEN_UNFROZEN)
    else:
        raise ValueError(stage_name)
    local_train = ReviewSet(train_frame, tokenizer, MAX_LENGTH)
    local_valid = ReviewSet(valid_frame, tokenizer, MAX_LENGTH)
    local_collator = DataCollatorWithPadding(tokenizer)
    trainer = WeightedTrainer(
        model=model,
        args=make_args(stage_name, learning_rate),
        train_dataset=local_train,
        eval_dataset=local_valid,
        data_collator=local_collator,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
        processing_class=tokenizer,
    )
    if groups is not None:
        def create_optimizer():
            trainer.optimizer = torch.optim.AdamW(groups, weight_decay=0.01)
            return trainer.optimizer
        trainer.create_optimizer = create_optimizer
    print(stage_name, "trainable parameters", sum(p.numel() for p in model.parameters() if p.requires_grad))
    return trainer

torch.manual_seed(SEED)
if stage == "head" and finished:
    stage = "last_block"
    model = AutoModelForSequenceClassification.from_pretrained(local_checkpoint)
    tokenizer = AutoTokenizer.from_pretrained(local_checkpoint)
    trainer = build_trainer(stage, model, tokenizer)
    print("head epoch is complete; starting the last encoder block")
    trainer.train()
else:
    model = AutoModelForSequenceClassification.from_pretrained(local_checkpoint)
    tokenizer = AutoTokenizer.from_pretrained(local_checkpoint)
    trainer = build_trainer(stage, model, tokenizer)
    print("resuming", stage, "from", local_checkpoint)
    trainer.train(resume_from_checkpoint=str(local_checkpoint))

block_dir = Path(DRIVE_DIR) / "runs" / f"{slug(MODEL_NAME)}--{stage}"
block_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(block_dir)
tokenizer.save_pretrained(block_dir)
print("saved before scoring", block_dir)

def score_restored(frame):
    holder = ReviewSet(frame, tokenizer, MAX_LENGTH)
    args = TrainingArguments(
        output_dir="/tmp/score-restored",
        per_device_eval_batch_size=BATCH_SIZE,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )
    scorer = Trainer(
        model=model,
        args=args,
        data_collator=DataCollatorWithPadding(tokenizer),
        processing_class=tokenizer,
    )
    prediction = scorer.predict(holder)
    return metric_bundle(prediction.label_ids, prediction.predictions.argmax(axis=-1))

validation_metrics = score_restored(valid_frame)
validation_metrics.update({"split": "validation", "stage": stage, "model_name": MODEL_NAME})
(block_dir / "metrics.json").write_text(json.dumps(validation_metrics, indent=2))
best_dir = Path(DRIVE_DIR) / "best" / slug(MODEL_NAME)
best_dir.mkdir(parents=True, exist_ok=True)
for item in block_dir.iterdir():
    if item.is_file():
        shutil.copy2(item, best_dir / item.name)
test_metrics = score_restored(test_frame)
test_metrics.update({"split": "test", "stage": stage, "model_name": MODEL_NAME})
(block_dir / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))
(best_dir / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))
print("validation", {key: validation_metrics[key] for key in ("accuracy", "precision", "recall", "f1")})
print("test", {key: test_metrics[key] for key in ("accuracy", "precision", "recall", "f1")})
print("run the MLflow push cell again before the next Colab shutdown")
